# Cross-Validation Compute Task

This notebook demonstrates how to evaluate estimation parameters using the `geostatistics/cross-validation` compute task in `evo-compute`.

Cross-validation holds samples out of a source dataset, re-estimates them at their own locations from the remaining samples, writes the estimates (and optional signed errors and kriging diagnostics) back onto the source object, and returns summary statistics.

Two tagged unions configure each run:

- **Hold-out strategy (`validation`)**:
  - `LeaveOneSampleOut()` — holds out each sample and any coincident samples.
  - `LeaveOneDrillholeOut()` — holds out every sample sharing one drillhole ID (requires a `DownholeIntervals` source).
- **Estimator (`estimator`)**:
  - `KrigingEstimator` — ordinary or simple kriging with a variogram and optional `KrigingDiagnostics`.
  - `IDWEstimator` — inverse-distance weighting with a positive `power`.
  - `KNNEstimator` — k-nearest-neighbour arithmetic mean.

## Authentication

Authenticate with Seequent Evo using `ServiceManagerWidget`:

In [ ]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(
    client_id="your-client-id", cache_location="./notebook-data"
).login()

In [ ]:
# Load the widgets extension for rich HTML display
%load_ext evo.widgets

## Example 1: Sample Cross-Validation (`LeaveOneSampleOut`)

Create a synthetic `PointSet` and `Variogram`, then compare kriging, IDW, and KNN using leave-one-sample-out cross-validation.

In [ ]:
import uuid

import numpy as np
import pandas as pd
from evo.objects.typed import (
    Ellipsoid,
    EllipsoidRanges,
    EpsgCode,
    PointSet,
    PointSetData,
    Rotation,
    SphericalStructure,
    Variogram,
    VariogramData,
)

rng = np.random.default_rng(42)
n_samples = 120
x = rng.uniform(0.0, 500.0, n_samples)
y = rng.uniform(0.0, 500.0, n_samples)
z = rng.uniform(0.0, 100.0, n_samples)
grade = 1.2 + 0.002 * x + 0.001 * y + rng.normal(0.0, 0.25, n_samples)
domain = np.where(x < 250.0, "West", "East")

sample_points = await PointSet.create(
    manager,
    PointSetData(
        name=f"Cross-Validation Samples - {uuid.uuid4()}",
        coordinate_reference_system=EpsgCode(32632),
        locations=pd.DataFrame({"x": x, "y": y, "z": z, "grade": grade, "domain": pd.Categorical(domain)}),
    ),
)

variogram = await Variogram.create(
    manager,
    VariogramData(
        name=f"Grade Variogram - {uuid.uuid4()}",
        sill=0.20,
        nugget=0.03,
        is_rotation_fixed=True,
        modelling_space="data",
        data_variance=0.20,
        structures=[
            SphericalStructure(
                contribution=0.17,
                anisotropy=Ellipsoid(
                    ranges=EllipsoidRanges(major=200.0, semi_major=150.0, minor=80.0),
                    rotation=Rotation(dip_azimuth=30.0, dip=15.0, pitch=0.0),
                ),
            )
        ],
        attribute="grade",
        domain="all",
    ),
)

sample_points

### Run Kriging, IDW, and KNN Sample Cross-Validation

Define one `CrossValidationParameters` per estimator and submit all three concurrently with `run(..., preview=True)`.
For kriging, request per-sample `KrigingDiagnostics` alongside the estimate and signed error (`estimate - true`).

In [ ]:
from evo.compute.tasks import KrigingDiagnostics, SearchNeighborhood, run
from evo.compute.tasks.geostatistics.cross_validation import (
    CrossValidationOutputs,
    CrossValidationParameters,
    IDWEstimator,
    KNNEstimator,
    KrigingEstimator,
    LeaveOneSampleOut,
)

search = SearchNeighborhood(
    ellipsoid=variogram.get_ellipsoid().scaled(1.5),
    max_samples=20,
    min_samples=3,
)

kriging_cv = CrossValidationParameters(
    source=sample_points.attributes["grade"],
    validation=LeaveOneSampleOut(),
    estimator=KrigingEstimator(
        variogram=variogram,
        neighborhood=search,
        diagnostics=KrigingDiagnostics(
            kriging_variance="ok_cv_kv",
            slope_of_regression="ok_cv_sor",
            num_samples="ok_cv_ns",
        ),
    ),
    outputs=CrossValidationOutputs(estimate="ok_cv_est", error="ok_cv_err"),
)

idw_cv = CrossValidationParameters(
    source=sample_points.attributes["grade"],
    validation=LeaveOneSampleOut(),
    estimator=IDWEstimator(neighborhood=search, power=2.0),
    outputs=CrossValidationOutputs(estimate="idw_cv_est", error="idw_cv_err"),
)

knn_cv = CrossValidationParameters(
    source=sample_points.attributes["grade"],
    validation=LeaveOneSampleOut(),
    estimator=KNNEstimator(neighborhood=search),
    outputs=CrossValidationOutputs(estimate="knn_cv_est", error="knn_cv_err"),
)

sample_results = await run(manager, [kriging_cv, idw_cv, knn_cv], preview=True)
kriging_result, idw_result, knn_result = sample_results
print(kriging_result)

### Compare Summary Statistics Across Estimators

Every result carries `result.statistics` over the samples that produced a valid estimate (`valid_count`).
Kriging additionally returns `result.kriging_statistics` with standardized error metrics (`mean_standardized_error` ideally near `0`, `standardized_error_variance` ideally near `1`).

In [ ]:
summary_rows = []
for label, res in [("Ordinary Kriging", kriging_result), ("IDW (p=2)", idw_result), ("KNN", knn_result)]:
    s = res.statistics
    ks = res.kriging_statistics
    summary_rows.append(
        {
            "Estimator": label,
            "Evaluated": s.evaluated_count,
            "Valid": s.valid_count,
            "Invalid": s.invalid_count,
            "Mean Error": s.mean_error,
            "MAE": s.mean_absolute_error,
            "RMSE": s.root_mean_squared_error,
            "Correlation": s.correlation,
            "Regression Slope": s.regression_slope,
            "Std Error Var": ks.standardized_error_variance if ks is not None else None,
        }
    )

pd.DataFrame(summary_rows).set_index("Estimator")

In [ ]:
# Inspect the per-sample estimates, signed errors, and kriging diagnostics as a DataFrame.
# Samples that did not satisfy the search constraints carry NaN in the estimate and error columns.
sample_df = await kriging_result.to_dataframe(
    "grade",
    kriging_result.estimate_name,
    kriging_result.error_name,
    *(attr.name for attr in kriging_result.diagnostics.values()),
)
sample_df.head()

In [ ]:
import plotly.express as px

valid_df = sample_df.dropna(subset=[kriging_result.estimate_name])
fig = px.scatter(
    valid_df,
    x=kriging_result.estimate_name,
    y="grade",
    title="Leave-One-Sample-Out Cross-Validation: True vs Estimated Grade",
    labels={kriging_result.estimate_name: "Estimated Grade", "grade": "True Grade"},
)
bounds = [
    min(valid_df["grade"].min(), valid_df[kriging_result.estimate_name].min()),
    max(valid_df["grade"].max(), valid_df[kriging_result.estimate_name].max()),
]
fig.add_scatter(x=bounds, y=bounds, mode="lines", name="1:1 Reference", line=dict(dash="dash", color="gray"))
fig.show()

## Example 2: Drillhole Cross-Validation (`LeaveOneDrillholeOut`)

When samples come from drillholes (`DownholeIntervals`), nearby intervals along the same hole strongly correlate with each other.
`LeaveOneDrillholeOut()` holds out an entire drillhole at a time and re-estimates all of its intervals strictly from the remaining holes.

In [ ]:
from evo.objects.typed import DownholeIntervals, DownholeIntervalsData

hole_frames = []
collar_xy = [(100.0, 100.0), (180.0, 120.0), (140.0, 200.0), (240.0, 190.0), (200.0, 280.0), (300.0, 260.0)]
n_intervals = 15

for idx, (cx, cy) in enumerate(collar_xy, start=1):
    hole_id = f"DH{idx:03d}"
    depths = np.linspace(0.0, 75.0, n_intervals + 1)
    from_d, to_d = depths[:-1], depths[1:]
    mid_d = 0.5 * (from_d + to_d)
    hole_grade = 1.0 + 0.002 * cx + 0.0015 * cy - 0.004 * mid_d + rng.normal(0.0, 0.15, n_intervals)
    hole_frames.append(
        pd.DataFrame(
            {
                "hole_id": hole_id,
                "from": from_d,
                "to": to_d,
                "x_start": np.full(n_intervals, cx),
                "y_start": np.full(n_intervals, cy),
                "z_start": -from_d,
                "x_end": np.full(n_intervals, cx),
                "y_end": np.full(n_intervals, cy),
                "z_end": -to_d,
                "x_mid": np.full(n_intervals, cx),
                "y_mid": np.full(n_intervals, cy),
                "z_mid": -mid_d,
                "grade": hole_grade,
            }
        )
    )

intervals_df = pd.concat(hole_frames, ignore_index=True)
intervals_df["hole_id"] = pd.Categorical(intervals_df["hole_id"])

downhole = await DownholeIntervals.create(
    manager,
    DownholeIntervalsData(
        name=f"Cross-Validation Drillholes - {uuid.uuid4()}",
        coordinate_reference_system=EpsgCode(32632),
        intervals=intervals_df,
        is_composited=True,
        depth_unit="m",
    ),
)

downhole

### Run Leave-One-Drillhole-Out Cross-Validation

Configure per-drillhole limits on `SearchNeighborhood` (`max_samples_per_drillhole` and `max_drillholes`) and request `num_drillholes` in `KrigingDiagnostics` to verify how many external holes contributed to each interval's estimate.

In [ ]:
from evo.compute.tasks.geostatistics.cross_validation import LeaveOneDrillholeOut

drillhole_search = SearchNeighborhood(
    ellipsoid=Ellipsoid(ranges=EllipsoidRanges(major=300.0, semi_major=300.0, minor=120.0)),
    max_samples=16,
    min_samples=4,
    max_samples_per_drillhole=4,
    max_drillholes=4,
)

drillhole_cv_params = CrossValidationParameters(
    source=downhole.attributes["grade"],
    validation=LeaveOneDrillholeOut(),
    estimator=KrigingEstimator(
        variogram=variogram,
        neighborhood=drillhole_search,
        diagnostics=KrigingDiagnostics(
            kriging_variance=True,
            num_drillholes=True,
            num_samples=True,
        ),
    ),
    outputs=CrossValidationOutputs(estimate="dh_cv_est", error="dh_cv_err"),
)

drillhole_result = await run(manager, drillhole_cv_params, preview=True)
print(drillhole_result)

In [ ]:
# Load interval estimates, errors, and contributing-drillhole counts by hole
dh_df = await drillhole_result.to_dataframe()
dh_df[["hole_id", "from", "to", "grade", "dh_cv_est", "dh_cv_err", "KV", "NDh", "NS"]].head(10)